In [48]:
import pandas as pd

# Load AIS data
ais_data = pd.read_csv('filtered_ais_data.csv')

# Convert BaseDateTime and COG to appropriate formats
ais_data['BaseDateTime'] = pd.to_datetime(ais_data['BaseDateTime'])
ais_data['COG'] = pd.to_numeric(ais_data['COG'], errors='coerce')  # Convert to float
ais_data = ais_data.dropna(subset=['COG'])  # Remove NaN values

# Sort by MMSI and BaseDateTime
ais_data = ais_data.sort_values(['MMSI', 'BaseDateTime'])

def detect_anomalies(df):
    anomalies = []
    unique_vessels = df['MMSI'].unique()
    flagged_vessels = set()

    for vessel in unique_vessels:
        if vessel in flagged_vessels:
            continue

        vessel_data = df[df['MMSI'] == vessel].reset_index(drop=True)

        for i in range(1, len(vessel_data)):
            previous_speed = vessel_data['SOG'].iloc[i - 1]
            current_speed = vessel_data['SOG'].iloc[i]
            speed_change = abs(current_speed - previous_speed)

            # Handle circular COG changes correctly
            cog_diff = abs(vessel_data['COG'].iloc[i] - vessel_data['COG'].iloc[i - 1])
            course_change = min(cog_diff, 360 - cog_diff)  # Handle wrap-around at 360°

            # Detect speed anomalies (>50% change relative to previous speed)
            if previous_speed > 2 and speed_change > 0.5 * previous_speed:
                anomalies.append(vessel_data.iloc[i])
                flagged_vessels.add(vessel)
                #break  # Stop checking further for this vessel

            # Detect course anomalies (>45° change at any speed)
            elif course_change > 45:
                anomalies.append(vessel_data.iloc[i])
                flagged_vessels.add(vessel)
                #break  # Stop checking further for this vessel

    return pd.DataFrame(anomalies)

# Run anomaly detection
anomalies_df = detect_anomalies(ais_data)
print("Anomalies detected:")
print(anomalies_df)


Anomalies detected:
        MMSI        BaseDateTime       LAT        LON  SOG    COG  Heading  \
2  235096399 2024-01-01 00:03:31  27.78969  -97.39082  3.0  112.1      511   
3  235096399 2024-01-01 00:06:03  27.78969  -97.39082  0.2  109.9      511   
3  248727000 2024-01-01 00:03:18  32.02417  -81.04617  0.1  212.0      328   
4  248727000 2024-01-01 00:05:37  32.02417  -81.04617  0.1  151.0      328   
1  316001252 2024-01-01 00:01:15  48.68902 -123.40947  0.0  241.2      243   
2  316001252 2024-01-01 00:02:24  48.68902 -123.40947  0.0  132.7      243   

            VesselName         IMO CallSign  VesselType  Status  Length  \
2              ASSINIE  IMO0000000    2GDG3          37     NaN       6   
3              ASSINIE  IMO0000000    2GDG3          37     NaN       6   
3          AMORAZUR II  IMO8981664  9HB2042          37     0.0      45   
4          AMORAZUR II  IMO8981664  9HB2042          37     0.0      45   
1  QUEEN OF CUMBERLAND  IMO9009360   VG2029          60   

In [49]:
# Install necessary packages
!pip install geemap
!pip install earthengine-api
# Install Earth Engine and Geemap
!pip install earthengine-api geemap


# Authenticate and initialize Earth Engine
import ee
import geemap

# Trigger the authentication
ee.Authenticate()

# Initialize the Earth Engine API
ee.Initialize(project='ee-oilspilldetection-by-uday')


In [50]:
def plot_histogram(image, region, title):
    try:
        array = image.reduceRegion(
            reducer=ee.Reducer.toList(),
            geometry=region,
            scale=30,
            maxPixels=1e9
        ).get('VV')

        # Check if array is empty
        if array is None:
            print("No VV data available for histogram plot.")
            return

        array = np.array(array.getInfo())
        plt.hist(array, bins=50, color='blue', alpha=0.7)
        plt.title(title)
        plt.xlabel('VV (dB)')
        plt.ylabel('Frequency')
        plt.grid()
        plt.show()
    except Exception as e:
        print(f"Error plotting histogram: {e}")

In [51]:
from datetime import datetime

# Function to parse date from BaseDateTime
def extract_date(base_date_time):
    # Convert to string if it’s a Timestamp
    if isinstance(base_date_time, pd.Timestamp):
        base_date_time = base_date_time.strftime('%Y-%m-%dT%H:%M:%S')
    else:
        base_date_time = str(base_date_time)

    # Parse the date
    date_time_obj = datetime.strptime(base_date_time, '%Y-%m-%dT%H:%M:%S')
    return date_time_obj.strftime('%Y-%m-%d')


In [52]:
def preprocess_image_for_model(image_np):
    """ Preprocess the NumPy array image to match model input requirements """
    # Resize the image to (256, 256)
    resized_image = cv2.resize(image_np, (256, 256))
    
    # Check the number of channels
    if resized_image.shape[-1] == 2:
        # Convert 2-channel input to 3-channel by duplicating one of the channels
        resized_image = np.repeat(resized_image[:, :, :1], 3, axis=-1)
    elif resized_image.shape[-1] == 1:
        # Convert 1-channel input to 3-channel
        resized_image = np.repeat(resized_image, 3, axis=-1)
    elif resized_image.shape[-1] != 3:
        raise ValueError(f"Unexpected number of channels: {resized_image.shape[-1]}")
    
    # Add a batch dimension
    return np.expand_dims(resized_image, axis=0)


In [53]:
import tensorflow as tf

# Define jaccard_coef if not already defined
def jaccard_coef(y_true, y_pred):
    y_true_flatten = tf.reshape(y_true, [-1])
    y_pred_flatten = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_flatten * y_pred_flatten)
    sum_ = tf.reduce_sum(y_true_flatten) + tf.reduce_sum(y_pred_flatten)
    smooth = 1e-6  # To avoid division by zero
    jac = (intersection + smooth) / (sum_ - intersection + smooth)
    return jac


In [54]:
from tensorflow.keras.models import load_model
import os

# Ensure segmentation_models uses TensorFlow backend
os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

# Load the models
hybrid_model = load_model(r"C:\Users\Uday Kiran Ambati\OneDrive\Desktop\major_project\Code\saved_models\hybrid_model.h5")
unet_model = load_model(r"C:\Users\Uday Kiran Ambati\OneDrive\Desktop\major_project\Code\saved_models\U_net.h5")
deeplab_model = load_model(
    r"C:\Users\Uday Kiran Ambati\OneDrive\Desktop\major_project\Code\saved_models\deeplab_model.h5",
    custom_objects={
        "jaccard_coef": jaccard_coef,  # Ensure this function is defined in your script
        "DiceLoss": sm.losses.DiceLoss(),  # Initialize the DiceLoss object
        "CategoricalFocalLoss": sm.losses.CategoricalFocalLoss()  # Initialize the CategoricalFocalLoss object
    }
)


In [55]:
def preprocess_image_for_model(image_np):
    """ Preprocess the NumPy array image to match model input requirements """
    # Remove any extra dimensions if present (e.g., shape (256, 256, 1, 6) -> (256, 256, 2))
    if image_np.ndim == 4:
        image_np = image_np[:, :, :, 0]

    # Resize to 256x256
    resized_image = cv2.resize(image_np, (256, 256))

    # Ensure the image has three channels (duplicate one if necessary)
    if resized_image.shape[-1] == 2:
        resized_image = np.repeat(resized_image[:, :, :1], 3, axis=-1)
    elif resized_image.shape[-1] == 1:
        resized_image = np.repeat(resized_image, 3, axis=-1)

    # Add batch dimension
    return np.expand_dims(resized_image, axis=0)

In [56]:
def confirm_oil_spill(hybrid_model, unet_model, deeplab_model, image):
    """ 
    Confirm oil spill detection using Hybrid, UNet, and DeepLabV3+ models. 
    Requires agreement among all models.
    """
    processed_image = preprocess_image_for_model(image)

    processed_image = preprocess_image_for_model(image)
        
        # Get predictions from both models
    unet_prediction = unet_model.predict(processed_image)
    deeplab_prediction = deeplab_model.predict(processed_image)
    hybrid_prediction = hybrid_model.predict(processed_image)

    # Assuming '1' corresponds to oil spill for both models
    unet_result = np.argmax(unet_prediction) == 1
    deeplab_result = np.argmax(deeplab_prediction) == 1
    hybrid_result = np.argmax(hybrid_prediction) == 1

    # Confirm only if both models agree
    #return unet_result 

    # Confirm only if all models agree
    return hybrid_result and unet_result and deeplab_result


In [57]:
from tensorflow.keras.models import load_model
import geemap
import ee
import numpy as np
import cv2
from IPython.display import display

def process_anomaly(row):
    lat = row['LAT']
    lon = row['LON']
    base_date_time = row['BaseDateTime']
    date = extract_date(base_date_time)
    anomaly_date = ee.Date(date)
    print(f"Processing anomaly at {lat}, {lon} on {date}")

    roi = ee.Geometry.Point(lon, lat).buffer(10000).bounds()
    detected_spills = []

    # Sentinel-1 collection
    sen1 = ee.ImageCollection("COPERNICUS/S1_GRD") \
        .filterDate(anomaly_date, anomaly_date.advance(1, 'day')) \
        .filterBounds(roi) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW'))

    if sen1.size().getInfo() == 0:
        print("No Sentinel-1 VV or VH images available. Trying Sentinel-2.")
        sen1 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
            .filterDate(anomaly_date, anomaly_date.advance(1, 'day')) \
            .filterBounds(roi) \
            .filter(ee.Filter.eq('CLOUDY_PIXEL_PERCENTAGE', 0))

        if sen1.size().getInfo() == 0:
            print("No Sentinel-2 images available for this date and location.")
            return None

        sen1_image = sen1.select(['B4', 'B8']).mosaic()
    else:
        sen1_image = sen1.select(['VV', 'VH']).mosaic()

    despeckled = sen1_image.focal_mean(100, 'square', 'meters')
    vv_dark = despeckled.select('VV').lt(-22)
    vh_dark = despeckled.select('VH').lt(-18)
    oil_spill_detection = vv_dark.And(vh_dark)

    mask = oil_spill_detection.updateMask(oil_spill_detection)
    area = mask.multiply(ee.Image.pixelArea().divide(1e6))
    oil_spill_area = ee.Number(
        area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=roi,
            scale=100
        ).values().get(0)
    ).getInfo()

    if oil_spill_area > 0:
        print(f"Detected Oil Spill Area (sq. km): {oil_spill_area}")

        despeckled_np = geemap.ee_to_numpy(despeckled, bands=['VV', 'VH'], region=roi, scale=100)

        # Ensure image preprocessing for models
        #processed_image = preprocess_image_for_model(despeckled_np)

        # Confirm using all three models
        if confirm_oil_spill(hybrid_model, unet_model, deeplab_model, despeckled_np):
            detected_spills.append(oil_spill_area)

            # oil_spill_vis_params = {
            #     'min': 0,
            #     'max': 1,
            #     'palette': ['blue', 'cyan', 'yellow', 'red']
            # }

            # Visualization
            Map = geemap.Map()
            Map.centerObject(roi)
            Map.addLayer(sen1_image, {'bands': ['VV', 'VH'], 'min': -30, 'max': 0}, f'Sentinel-1 Image {anomaly_date.getInfo()}')
            Map.addLayer(despeckled.clip(roi), {}, 'Despeckled Image', False)
            Map.addLayer(oil_spill_detection.clip(roi), {}, 'Oil Spill Detection', False)
            oil_spill_vector = mask.reduceToVectors(
                geometry=roi,
                scale=100
            )
            Map.addLayer(oil_spill_vector, {}, 'Oil Spill Vector', False)

    if detected_spills:
        print(f"Oil spills detected over the days: {detected_spills}")
        Map.addLayerControl()
        return Map
    else:
        print("No significant oil spills detected across the checked dates.")
        return None


In [58]:
# Iterate over each anomaly in the DataFrame and process
for index, row in anomalies_df.iterrows():
    result_map = process_anomaly(row)
    if result_map:
        display(result_map)  # This displays the map in interactive environments

Processing anomaly at 27.78969, -97.39082 on 2024-01-01
No Sentinel-1 VV or VH images available. Trying Sentinel-2.
No Sentinel-2 images available for this date and location.
Processing anomaly at 27.78969, -97.39082 on 2024-01-01
No Sentinel-1 VV or VH images available. Trying Sentinel-2.
No Sentinel-2 images available for this date and location.
Processing anomaly at 32.02417, -81.04617 on 2024-01-01
Detected Oil Spill Area (sq. km): 17.5633537601677
1/1 [==============================] - 2s 2s/step
No significant oil spills detected across the checked dates.
Processing anomaly at 32.02417, -81.04617 on 2024-01-01
Detected Oil Spill Area (sq. km): 17.5633537601677
1/1 [==============================] - 0s 336ms/step
No significant oil spills detected across the checked dates.
Processing anomaly at 48.68902, -123.40947 on 2024-01-01
Detected Oil Spill Area (sq. km): 204.79842854767278
1/1 [==============================] - 0s 343ms/step
No significant oil spills detected across the ch